# Detection & Classification using machine learning

HOG + color moments + brightness features, linear SVM, selective search proposals.
Run from the repository root with `dataset/{train,val,test}` in place (see README).

## Building the dataset

In [ ]:
import warnings

seed = 42
train_images, train_labels = "dataset/train/images/", "dataset/train/labels/"
val_images, val_labels = "dataset/val/images/", "dataset/val/labels/"
test_images = "dataset/test/images/"
aug_images, aug_labels = "dataset/train/images_aug/", "dataset/train/labels_aug/"

train_path = "dataset/train/cropped_images/"
aug_train_path = "dataset/train/aug_cropped_images/"
val_path = "dataset/val/cropped_images/"

standard_size = (64, 64)
label_size_factor = 1
max_iter = 10000

warnings.filterwarnings("ignore", message="Applying `local_binary_pattern` to floating-point images")

#### Data augmentation

In [ ]:
from src.augment import ImageBatchProcessor

processor = ImageBatchProcessor(image_folder=train_images, label_folder=train_labels,
                                target_image_folder=aug_images, target_label_folder=aug_labels)
processor.augment_and_save(nb_augmentation=5)

#### Image cropping and negative examples generation

In [ ]:
from src.crop import crop_images_from_folder

crop_images_from_folder(train_images, train_labels, train_path)
crop_images_from_folder(aug_images, aug_labels, aug_train_path)
crop_images_from_folder(val_images, val_labels, val_path)

#### Dataset building and training

The validation set is never used for training: it is only used to evaluate each round.

In [ ]:
from collections import Counter

from src.dataset import dataset
from src.model import model


def train_round(name):
    train_data = dataset(img_dir=train_path, augment_path=aug_train_path, label_size_factor=label_size_factor,
                         standard_size=standard_size, seed=seed)
    val_data = dataset(val_path, train=False, standard_size=standard_size)
    print("Train labels:", Counter(im.label for im in train_data.images))
    print("Val labels:", Counter(im.label for im in val_data.images))

    my_model = model(seed, standard_size=standard_size)
    print(f"Training {name} model with seed {seed}, max_iter {max_iter}, standard_size {standard_size}, label_size_factor {label_size_factor}")
    my_model.train_svm(train_data, max_iter=max_iter)
    my_model.evaluate(val_data)
    return my_model

## First model training

In [ ]:
my_model = train_round("first")

## Hard negative mining

Run detection on the **train** images and add every detection that overlaps no annotated box to the training set as `none`.

In [ ]:
from src.detection import detection_images_in_folder
from src.evaluation import load_detections, load_ground_truth
from src.negatives import get_negative_prediction

train_ground_truth = load_ground_truth(train_labels)


def mine_hard_negatives(my_model):
    detection_images_in_folder(train_images, my_model, "detections_train.csv", output_folder=None)
    get_negative_prediction(load_detections("detections_train.csv"), train_ground_truth, train_images, train_path)


mine_hard_negatives(my_model)

## Second model training

In [ ]:
my_model = train_round("second")
mine_hard_negatives(my_model)

## Third model training

In [ ]:
my_model = train_round("third")
my_model.save("src/models", "third_training")

## Final detection on the validation set

In [ ]:
from src.evaluation import evaluate_detections

detection_images_in_folder(val_images, my_model, "detections_val.csv", output_folder="output_images/val")
report = evaluate_detections(load_detections("detections_val.csv"), load_ground_truth(val_labels))

## Detection on the test set

In [ ]:
detection_images_in_folder(test_images, my_model, "detections_test.csv", output_folder="output_images/test")